In [24]:
import tkinter as tk
from tkinter import ttk, messagebox
import mysql.connector
import webbrowser
import matplotlib.pyplot as plt
from queue import PriorityQueue
from PIL import Image, ImageTk

# ================= DATABASE =================
try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="ai_system"
    )
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INT AUTO_INCREMENT PRIMARY KEY,
        username VARCHAR(50) UNIQUE,
        password VARCHAR(50),
        role VARCHAR(20)
    )
    """)
    conn.commit()
except:
    conn = None
    cursor = None

# ================= REGISTER (FIXED) =================
def register():
    win = tk.Toplevel()
    win.title("Register")
    win.geometry("300x250")
    win.configure(bg="#12121c")

    tk.Label(win, text="Username", bg="#12121c", fg="white").pack(pady=5)
    u = ttk.Entry(win)
    u.pack()

    tk.Label(win, text="Password", bg="#12121c", fg="white").pack(pady=5)
    p = ttk.Entry(win, show="*")
    p.pack()

    tk.Label(win, text="Role", bg="#12121c", fg="white").pack(pady=5)
    r = ttk.Entry(win)
    r.pack()

    def save_user():
        try:
            cursor.execute(
                "INSERT INTO users(username, password, role) VALUES (%s,%s,%s)",
                (u.get(), p.get(), r.get())
            )
            conn.commit()
            messagebox.showinfo("Success", "User Registered Successfully!")
            win.destroy()

        except mysql.connector.IntegrityError:
            messagebox.showerror("Error", "Username already exists")

        except:
            messagebox.showerror("Error", "Database error")

    ttk.Button(win, text="Register", command=save_user).pack(pady=10)

# ================= GRAPH =================
graph = {
    "A": {"B": 1, "C": 3},
    "B": {"D": 2, "E": 4},
    "C": {"F": 5},
    "D": {},
    "E": {"F": 1},
    "F": {}
}

# ================= SEARCH =================
def bfs(start, goal):
    queue = [(start, [start])]
    visited = set()
    while queue:
        node, path = queue.pop(0)
        if node == goal:
            return path
        if node not in visited:
            visited.add(node)
            for n in graph.get(node, {}):
                queue.append((n, path + [n]))

def dfs(start, goal):
    stack = [(start, [start])]
    visited = set()
    while stack:
        node, path = stack.pop()
        if node == goal:
            return path
        if node not in visited:
            visited.add(node)
            for n in graph.get(node, {}):
                stack.append((n, path + [n]))

def best_first(start, goal):
    pq = PriorityQueue()
    pq.put((0, start, [start]))
    while not pq.empty():
        cost, node, path = pq.get()
        if node == goal:
            return path
        for n, w in graph.get(node, {}).items():
            pq.put((w, n, path + [n]))

def heuristic(n):
    return {"A":5,"B":4,"C":3,"D":2,"E":1,"F":0}.get(n,0)

def a_star(start, goal):
    pq = PriorityQueue()
    pq.put((0, start, [start], 0))
    while not pq.empty():
        f, node, path, g = pq.get()
        if node == goal:
            return path
        for n, w in graph.get(node, {}).items():
            new_g = g + w
            pq.put((new_g + heuristic(n), n, path + [n], new_g))

# ================= MINIMAX GAME =================
def check_winner(b):
    win = [(0,1,2),(3,4,5),(6,7,8),
           (0,3,6),(1,4,7),(2,5,8),
           (0,4,8),(2,4,6)]
    for i,j,k in win:
        if b[i]==b[j]==b[k] and b[i]!=" ":
            return b[i]
    return None

def minimax(board, is_max):
    winner = check_winner(board)
    if winner == "X": return -1
    if winner == "O": return 1
    if " " not in board: return 0

    if is_max:
        best = -100
        for i in range(9):
            if board[i]==" ":
                board[i]="O"
                best = max(best, minimax(board, False))
                board[i]=" "
        return best
    else:
        best = 100
        for i in range(9):
            if board[i]==" ":
                board[i]="X"
                best = min(best, minimax(board, True))
                board[i]=" "
        return best

def best_move(board):
    best_val = -100
    move = -1
    for i in range(9):
        if board[i]==" ":
            board[i]="O"
            val = minimax(board, False)
            board[i]=" "
            if val > best_val:
                best_val = val
                move = i
    return move

def minimax_game():
    win = tk.Toplevel(bg="#12121c")
    win.title("🎮 Minimax Game")
    win.geometry("300x320")

    board = [" "]*9
    buttons = []

    def click(i):
        if board[i]==" ":
            board[i]="X"
            buttons[i].config(text="X")

            if check_winner(board):
                messagebox.showinfo("Game","You Win!")
                win.destroy()
                return

            ai = best_move(board)
            if ai!=-1:
                board[ai]="O"
                buttons[ai].config(text="O")

            if check_winner(board):
                messagebox.showinfo("Game","AI Wins!")
                win.destroy()

    for i in range(9):
        b = tk.Button(win, text=" ", width=5, height=2,
                      font=("Arial",20),
                      command=lambda i=i: click(i))
        b.grid(row=i//3, column=i%3)
        buttons.append(b)

# ================= CHART =================
def show_chart():
    cursor.execute("SELECT role, COUNT(*) FROM users GROUP BY role")
    data = cursor.fetchall()

    roles = [x[0] for x in data]
    counts = [x[1] for x in data]

    plt.figure()
    plt.bar(roles, counts)
    plt.title("User Distribution")
    plt.show()

# ================= GPS =================
def real_navigation():
    win = tk.Toplevel(bg="#1e1e2f")
    win.title("🧭 Navigation")
    win.geometry("350x350")

    tk.Label(win, text="Start", fg="white", bg="#1e1e2f").pack()
    s = ttk.Entry(win)
    s.pack(pady=5)

    tk.Label(win, text="End", fg="white", bg="#1e1e2f").pack()
    e = ttk.Entry(win)
    e.pack(pady=5)

    algo = tk.StringVar(value="BFS")
    ttk.Combobox(win, textvariable=algo,
                 values=["BFS","DFS","A*","BestFS"]).pack(pady=5)

    result = tk.Label(win, text="", fg="cyan", bg="#1e1e2f")
    result.pack(pady=10)

    def find():
        start, end = s.get(), e.get()
        webbrowser.open(f"https://www.google.com/maps/dir/{start}/{end}")

        if algo.get()=="BFS":
            path = bfs(start,end)
        elif algo.get()=="DFS":
            path = dfs(start,end)
        elif algo.get()=="A*":
            path = a_star(start,end)
        else:
            path = best_first(start,end)

        result.config(text=f"Path: {path}")

    ttk.Button(win, text="Find Route", command=find).pack(pady=10)

# ================= ADMIN DASHBOARD =================
def admin_dashboard():
    win = tk.Toplevel(bg="#12121c")
    win.title("Admin Dashboard")
    win.geometry("900x500")

    sidebar = tk.Frame(win, bg="#2c2c3c", width=200)
    sidebar.pack(side="left", fill="y")

    main = tk.Frame(win, bg="#12121c")
    main.pack(side="right", expand=True, fill="both")

    def clear():
        for w in main.winfo_children():
            w.destroy()

    def home():
        clear()
        tk.Label(main, text="👑 Admin Panel",
                 fg="white", bg="#12121c",
                 font=("Arial",18)).pack(pady=20)

    def users():
        clear()
        cursor.execute("SELECT id, username, role FROM users")
        for row in cursor.fetchall():
            frame = tk.Frame(main, bg="#1e1e2f")
            frame.pack(fill="x")
            tk.Label(frame, text=str(row),
                     fg="white", bg="#1e1e2f").pack(side="left")
            tk.Button(frame, text="❌",
                      command=lambda r=row[0]: delete_user(r)).pack(side="right")

    def delete_user(uid):
        cursor.execute("DELETE FROM users WHERE id=%s", (uid,))
        conn.commit()
        users()

    tk.Button(sidebar, text="🏠 Home", command=home).pack(fill="x")
    tk.Button(sidebar, text="👥 Users", command=users).pack(fill="x")
    tk.Button(sidebar, text="🧭 GPS", command=real_navigation).pack(fill="x")
    tk.Button(sidebar, text="📊 Charts", command=show_chart).pack(fill="x")
    tk.Button(sidebar, text="🎮 Game", command=minimax_game).pack(fill="x")

    home()

# ================= LOGIN =================
def login():
    try:
        cursor.execute(
            "SELECT role FROM users WHERE username=%s AND password=%s",
            (user.get(), pwd.get())
        )
        r = cursor.fetchone()

        if r:
            if r[0]=="admin":
                admin_dashboard()
            else:
                real_navigation()
        else:
            messagebox.showerror("Error","Invalid Login")

    except:
        messagebox.showerror("DB Error","Connection Lost → Game Starting")
        minimax_game()

# ================= MAIN =================
root = tk.Tk()
root.title("AI Smart System")
root.geometry("400x300")
root.configure(bg="#12121c")

frame = tk.Frame(root, bg="#1e1e2f", padx=20, pady=20)
frame.place(relx=0.5, rely=0.5, anchor="center")

tk.Label(frame, text="Login",
         fg="white", bg="#1e1e2f",
         font=("Arial",16)).pack(pady=10)

user = ttk.Entry(frame)
user.pack(pady=5)

pwd = ttk.Entry(frame, show="*")
pwd.pack(pady=5)

ttk.Button(frame, text="Login", command=login).pack(pady=5)

# ✅ FIXED HERE
ttk.Button(frame, text="Register", command=register).pack()

root.mainloop()

In [18]:
!pip install mysql-connector-python

In [6]:
pip install matplotlib requests

Note: you may need to restart the kernel to use updated packages.
